In [ ]:
!pip install qiskit qiskit-optimization qiskit-algorithms pandas numpy
    # optional, for the interconnection graph picture:
!pip install networkx matplotlib qiskit -u


[optparse.groups]Usage:[/]   
  pip install \[options] <requirement specifier> \[package-index-options] ...
  pip install \[options] -r <requirements file> \[package-index-options] ...
  pip install \[options] [-e] <vcs project url> ...
  pip install \[options] [-e] <local project path> ...
  pip install \[options] <archive url/path> ...

no such option: -u


In [ ]:
"""
QUBO-based structured pruning of a ConvNeXT / SwinV2 model, solved with Qiskit.

PIPELINE
--------
    cost_loss_sensitivity.py   ->   cost_loss_table.csv   ->   THIS SCRIPT

This script takes the per-block sensitivity table you already produced
(columns: candidate, L_i, C_i, loss_increase_raw, accuracy_drop_raw, params, ...)
and decides WHICH blocks to prune by minimizing a QUBO (Quadratic Unconstrained
Binary Optimization) Hamiltonian on a quantum solver.

DECISION VARIABLES
------------------
    x_i = 1  ->  PRUNE block i      (remove / bypass it, save its params)
    x_i = 0  ->  KEEP  block i

OBJECTIVE (the QUBO)
--------------------
    minimize   H(x) = sum_i ( lambda * L_i - gamma * C_i ) * x_i        # linear: loss vs. savings
                    + sum_{i<j, same stage} J_ij * x_i * x_j            # quadratic: interconnection penalty

    * L_i (normalized loss increase) = "importance"  -> high means painful to prune.
    * C_i (normalized param count)   = "reward"      -> high means big savings.
    * lambda  weights accuracy-preservation vs. gamma which weights size-reduction.
    * J_ij    penalizes pruning two blocks that live in the SAME residual stage
              together, because residual blocks in one stage are interconnected and
              their damage is super-additive (a single-block sweep cannot see this).

WHY QUBO / QISKIT
-----------------
    Subset selection with pairwise interactions maps 1:1 onto an Ising Hamiltonian.
    Qiskit's QAOA searches for that Hamiltonian's ground state = the optimal prune
    mask. With only a handful of qubits we ALSO solve it exactly (classically) so we
    can confirm QAOA found the true optimum.

INSTALL
-------
    pip install qiskit qiskit-optimization qiskit-algorithms pandas numpy
    # optional, for the interconnection graph picture:
    pip install networkx matplotlib

RUN
---
    python qubo_prune.py --csv cost_loss_table.csv --lambda-loss 8 --gamma-cost 1 --coupling 0.6
    python qubo_prune.py --csv cost_loss_table.csv --sweep            # Pareto sweep over lambda
    python qubo_prune.py --csv cost_loss_table.csv --draw-graph       # save interconnection graph PNG
"""

from __future__ import annotations

import argparse
import itertools
import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# --- Qiskit optimization stack -------------------------------------------------
# qiskit-optimization builds the QuadraticProgram and converts QUBO -> Ising.
# qiskit-algorithms provides QAOA and the exact (NumPy) eigensolver.
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals
# Qiskit >= 1.0 removed the V1 `Sampler`; Qiskit 2.x ships only V2 primitives.
# StatevectorSampler is the local, exact replacement and QAOA accepts it.
from qiskit.primitives import StatevectorSampler


# ==============================================================================
# 1. Load the sensitivity table and parse the block topology
# ==============================================================================

# A candidate name looks like "stages.2.blocks.4" (ConvNeXT) or
# "layers.1.blocks.0" (SwinV2). We extract (stage_index, block_index) so we know
# which blocks share a residual stage and are therefore interconnected.
_BLOCK_RE = re.compile(r"^(?:stages|layers)\.(\d+)\.blocks\.(\d+)$")


@dataclass
class Block:
    name: str          # e.g. "stages.2.blocks.4"
    stage: int         # residual stage this block belongs to
    index: int         # block index within the stage
    L: float           # normalized loss increase  (importance)
    C: float           # normalized parameter cost  (savings reward)
    params: int        # raw parameter count
    acc_drop: float    # raw accuracy drop (for reporting only)


def load_blocks(csv_path: str) -> List[Block]:
    df = pd.read_csv(csv_path)
    required = {"candidate", "L_i", "C_i", "params"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    blocks: List[Block] = []
    for _, row in df.iterrows():
        m = _BLOCK_RE.match(str(row["candidate"]))
        if not m:
            # Skip names we cannot place in the stage/block topology.
            print(f"  [warn] could not parse topology from '{row['candidate']}', skipping")
            continue
        blocks.append(
            Block(
                name=str(row["candidate"]),
                stage=int(m.group(1)),
                index=int(m.group(2)),
                L=float(row["L_i"]),
                C=float(row["C_i"]),
                params=int(row["params"]),
                acc_drop=float(row.get("accuracy_drop_raw", np.nan)),
            )
        )
    # Stable, human-readable ordering: by stage then block index.
    blocks.sort(key=lambda b: (b.stage, b.index))
    return blocks


# ==============================================================================
# 2. Build the interconnection (coupling) graph
# ==============================================================================

def build_coupling(
    blocks: List[Block],
    coupling: float,
    adjacency_boost: float = 1.5,
) -> Dict[Tuple[int, int], float]:
    """
    Return J_ij for every pair of blocks that are interconnected.

    Interconnection rule (ConvNeXT / Swin):
        Two blocks are coupled iff they live in the SAME stage, because a stage
        is a single residual stream at one resolution/width. Pruning two blocks
        in the same stage damages that shared stream super-additively.

    Strength:
        J_ij = coupling                      for same-stage pairs
        J_ij = coupling * adjacency_boost    if their block indices are close
                                             (|delta index| <= 2)  -> even more
                                             correlated, penalize harder.

    Pairs in DIFFERENT stages are separated by a downsampling layer and are
    treated as (approximately) independent -> no coupling term.
    """
    J: Dict[Tuple[int, int], float] = {}
    for i, j in itertools.combinations(range(len(blocks)), 2):
        bi, bj = blocks[i], blocks[j]
        if bi.stage != bj.stage:
            continue
        strength = coupling
        if abs(bi.index - bj.index) <= 2:
            strength *= adjacency_boost
        J[(i, j)] = strength
    return J


# ==============================================================================
# 3. Assemble the QUBO as a Qiskit QuadraticProgram
# ==============================================================================

def build_qubo(
    blocks: List[Block],
    J: Dict[Tuple[int, int], float],
    lambda_loss: float,
    gamma_cost: float,
    protect_threshold: float = 0.5,
    protect_penalty: float = 100.0,
) -> QuadraticProgram:
    """
    Construct the QUBO:

        minimize  sum_i a_i x_i + sum_{i<j} J_ij x_i x_j
        with      a_i = lambda_loss * L_i - gamma_cost * C_i

    Hard protection:
        Any block whose importance L_i exceeds `protect_threshold` (e.g.
        stages.0.blocks.0 with L=1.0) is essentially un-prunable. We add a large
        positive penalty to its linear term so the minimizer always keeps it
        (x_i = 0). This is a soft-but-effectively-hard constraint that keeps the
        problem a clean unconstrained QUBO.
    """
    qp = QuadraticProgram(name="block_pruning")

    # One binary variable per candidate block.
    for b in blocks:
        qp.binary_var(name=b.name)

    # Linear coefficients a_i.
    linear: Dict[str, float] = {}
    for b in blocks:
        a_i = lambda_loss * b.L - gamma_cost * b.C
        if b.L >= protect_threshold:
            a_i += protect_penalty  # never prune a critical block
        linear[b.name] = a_i

    # Quadratic coefficients J_ij (interconnection penalties).
    quadratic: Dict[Tuple[str, str], float] = {
        (blocks[i].name, blocks[j].name): w for (i, j), w in J.items()
    }

    qp.minimize(linear=linear, quadratic=quadratic)
    return qp


# ==============================================================================
# 4. Solve: exact (classical, for ground truth) and QAOA (quantum)
# ==============================================================================

def solve_exact(qp: QuadraticProgram):
    """Brute-force ground state via NumPy eigensolver. Trivial for <~20 qubits."""
    solver = MinimumEigenOptimizer(NumPyMinimumEigensolver())
    return solver.solve(qp)


def solve_qaoa(qp: QuadraticProgram, reps: int = 3, maxiter: int = 250, seed: int = 42):
    """
    Solve with QAOA on the statevector Sampler primitive.

    reps (p) = number of QAOA layers; more layers -> better approximation,
    more parameters to optimize. COBYLA is a robust gradient-free optimizer
    for the classical outer loop.
    """
    algorithm_globals.random_seed = seed
    qaoa = QAOA(
        sampler=StatevectorSampler(),
        optimizer=COBYLA(maxiter=maxiter),
        reps=reps,
    )
    solver = MinimumEigenOptimizer(qaoa)
    return solver.solve(qp)


# ==============================================================================
# 5. Reporting
# ==============================================================================

def report(blocks: List[Block], result, title: str) -> Tuple[List[Block], List[Block]]:
    """Pretty-print the prune/keep decision and the realized trade-off."""
    x = result.variables_dict  # {block_name: 0.0 or 1.0}

    pruned = [b for b in blocks if x[b.name] > 0.5]
    kept = [b for b in blocks if x[b.name] <= 0.5]

    total_params = sum(b.params for b in blocks)
    saved_params = sum(b.params for b in pruned)
    pred_loss = sum(b.L for b in pruned)            # additive proxy (normalized)
    pred_acc_drop = sum(b.acc_drop for b in pruned) # additive proxy (raw)

    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")
    print(f"objective value : {result.fval:.4f}")
    print(f"\nPRUNE ({len(pruned)}):")
    for b in pruned:
        print(f"  - {b.name:<22s} params={b.params:>9,}  L_i={b.L:.4f}  C_i={b.C:.4f}")
    print(f"\nKEEP  ({len(kept)}):")
    for b in kept:
        flag = "  <-- protected (critical)" if b.L >= 0.5 else ""
        print(f"  - {b.name:<22s} params={b.params:>9,}  L_i={b.L:.4f}  C_i={b.C:.4f}{flag}")

    print(f"\nparams removed  : {saved_params:,} / {total_params:,} "
          f"({100.0 * saved_params / max(total_params, 1):.1f}% of candidate params)")
    print(f"pred. loss cost : {pred_loss:.4f}  (sum of normalized L_i over pruned)")
    if not np.isnan(pred_acc_drop):
        print(f"pred. acc drop  : {pred_acc_drop:.4f}  (additive proxy; real value needs re-eval)")
    return pruned, kept


def pareto_sweep(blocks: List[Block], J, gamma_cost: float, lambdas: List[float]):
    """Sweep lambda to expose the savings-vs-accuracy Pareto front."""
    print(f"\n{'=' * 70}\nPARETO SWEEP over lambda_loss (gamma_cost={gamma_cost})\n{'=' * 70}")
    print(f"{'lambda':>8} | {'#pruned':>7} | {'%params':>8} | {'pred_loss':>9} | pruned blocks")
    print("-" * 90)
    total_params = sum(b.params for b in blocks)
    for lam in lambdas:
        qp = build_qubo(blocks, J, lambda_loss=lam, gamma_cost=gamma_cost)
        res = solve_exact(qp)  # exact is cheap and deterministic for the sweep
        x = res.variables_dict
        pruned = [b for b in blocks if x[b.name] > 0.5]
        saved = sum(b.params for b in pruned)
        ploss = sum(b.L for b in pruned)
        names = ", ".join(b.name.replace("stages.", "s").replace(".blocks.", "b") for b in pruned)
        print(f"{lam:>8.2f} | {len(pruned):>7} | {100*saved/total_params:>7.1f}% | "
              f"{ploss:>9.4f} | {names}")


def draw_graph(blocks: List[Block], J, pruned_names, out_path: str = "interconnection_graph.png"):
    """Optional: visualize the interconnection graph, colored by prune/keep."""
    try:
        import matplotlib.pyplot as plt
        import networkx as nx
    except ImportError:
        print("  [skip] install networkx + matplotlib to draw the graph")
        return

    G = nx.Graph()
    for b in blocks:
        G.add_node(b.name, stage=b.stage)
    for (i, j), w in J.items():
        G.add_edge(blocks[i].name, blocks[j].name, weight=w)

    pos = {b.name: (b.index, -b.stage) for b in blocks}  # stage = row, index = column
    colors = ["#d9534f" if b.name in pruned_names else "#5cb85c" for b in blocks]

    plt.figure(figsize=(9, 6))
    nx.draw_networkx_edges(G, pos, width=[2 * G[u][v]["weight"] for u, v in G.edges()],
                           edge_color="#888", alpha=0.6)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=1700, edgecolors="black")
    nx.draw_networkx_labels(G, pos, labels={b.name: b.name.replace("stages.", "s").replace(".blocks.", "\nb") for b in blocks},
                            font_size=8)
    plt.title("Block interconnection graph  (red = prune, green = keep; edges = same-stage coupling)")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    print(f"  saved interconnection graph -> {out_path}")


In [ ]:
"""
Pruning QUBO solved as an explicit Variational Quantum Algorithm (VQA),
following the structure of IBM's "Variational Algorithm Design" course.

    https://quantum.cloud.ibm.com/learning/en/courses/variational-algorithm-design

Where qubo_prune.py used the high-level MinimumEigenOptimizer(QAOA(...)) wrapper,
THIS script exposes the five course building blocks explicitly so you can see and
tune each one:

    1. REFERENCE STATE  -> uniform superposition H^(x)n  (built into QAOAAnsatz)
    2. ANSATZ           -> QAOAAnsatz constructed FROM our pruning Hamiltonian
    3. COST FUNCTION    -> <psi(theta)| H |psi(theta)>  via the Estimator primitive
    4. OPTIMIZER        -> classical scipy COBYLA loop over the QAOA angles
    5. BOOTSTRAPPING    -> initial_point for the angles (warm start)

The Hamiltonian is exactly the Ising operator we derived by hand:
    H = sum_i h_i Z_i + sum_{i<j} J^Z_ij Z_i Z_j + offset
We let qiskit-optimization perform the x_i = (1 - z_i)/2 substitution for us, then
feed the resulting SparsePauliOp straight into QAOAAnsatz -- the same pattern the
course's Max-Cut / Cost Functions lesson uses.

INSTALL
-------
    pip install qiskit qiskit-optimization scipy pandas numpy
    # to run on real hardware instead of local simulation:
    #   pip install qiskit-ibm-runtime

RUN
---
    python qubo_prune_vqa.py --csv cost_loss_table.csv --reps 3 --maxiter 300
"""

from __future__ import annotations

import argparse
import numpy as np
from scipy.optimize import minimize

# --- reuse the problem definition we already wrote in qubo_prune.py -----------
# (load the CSV, build the same-stage coupling graph, and assemble the QUBO)
# NOTE: load_blocks, build_coupling, and build_qubo are defined in the cell
# above, so we use them directly instead of importing from a qubo_prune module.

# --- VQA building blocks, the course way --------------------------------------
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
# V2 primitives: local, exact-statevector implementations. Swap for
# qiskit_ibm_runtime EstimatorV2/SamplerV2 to run on real QPUs.
from qiskit.primitives import StatevectorEstimator, StatevectorSampler


# ==============================================================================
# Step 0: QUBO  ->  Ising Hamiltonian (the cost operator / observable)
# ==============================================================================

def qubo_to_hamiltonian(qp):
    """
    Convert our QuadraticProgram into the Ising observable H and a scalar offset.

    QuadraticProgramToQubo applies the x_i = (1 - z_i)/2 substitution and returns
    a SparsePauliOp whose:
        - weight-1 'Z' terms have coefficients h_i,
        - weight-2 'ZZ' terms have coefficients J^Z_ij,
        - offset is the constant energy shift C.
    This is precisely the mapping we derived analytically.
    """
    qubo = QuadraticProgramToQubo().convert(qp)
    hamiltonian, offset = qubo.to_ising()
    return hamiltonian, offset, qubo


def build_hamiltonian_by_hand(blocks, J, lambda_loss, gamma_cost,
                              protect_threshold=0.5, protect_penalty=100.0):
    """
    Build the SAME Ising observable directly from the formulas we derived,
    using the lesson's `SparsePauliOp.from_list([("ZZII", w), ...])` idiom.

    From  a_i = lambda*L_i - gamma*C_i  and  b_ij = J_ij  (same-stage coupling):

        h_i     = -a_i/2  -  (1/4) * sum_{j != i} b_ij        # single-Z 'field'
        J^Z_ij  =  b_ij / 4                                   # ZZ coupling
        offset  =  sum_i a_i/2  +  sum_{i<j} b_ij/4           # constant

    Compared with the course's Max-Cut Hamiltonian (ZZ terms only), ours adds the
    single-Z field terms h_i -- pruning is "Max-Cut WITH local biases".
    """
    n = len(blocks)

    # a_i (with hard protection on critical blocks)
    a = []
    for b in blocks:
        a_i = lambda_loss * b.L - gamma_cost * b.C
        if b.L >= protect_threshold:
            a_i += protect_penalty
        a.append(a_i)

    # b_ij from the coupling dict (keys are (i, j) index pairs into `blocks`)
    def z_string(positions):
        # Qiskit is little-endian: qubit i -> position (n-1-i) in the string.
        chars = ["I"] * n
        for i in positions:
            chars[n - 1 - i] = "Z"
        return "".join(chars)

    terms = []
    offset = 0.0
    h = [0.0] * n

    # couplings -> ZZ terms + their leakage into the single-Z fields
    for (i, j), b_ij in J.items():
        terms.append((z_string([i, j]), b_ij / 4.0))
        h[i] -= b_ij / 4.0
        h[j] -= b_ij / 4.0
        offset += b_ij / 4.0

    # linear a_i -> single-Z fields + constant
    for i in range(n):
        h[i] += -a[i] / 2.0
        offset += a[i] / 2.0

    for i in range(n):
        if abs(h[i]) > 1e-12:
            terms.append((z_string([i]), h[i]))

    hamiltonian = SparsePauliOp.from_list(terms, num_qubits=n)
    return hamiltonian.simplify(), offset


# ==============================================================================
# Step 3: the COST FUNCTION (course signature)
# ==============================================================================

def cost_func(params, ansatz, hamiltonian, estimator):
    """
    Energy expectation <psi(params)| H |psi(params)>.

    This is the black-box oracle the classical optimizer queries. Identical in
    form to VQE -- QAOA is just VQE with a problem-structured ansatz.
    """
    pub = (ansatz, hamiltonian, params)          # Primitive Unified Bloc (V2 API)
    result = estimator.run([pub]).result()
    energy = result[0].data.evs
    return float(energy)


# ==============================================================================
# Step 5 + readout: run the variational loop, then SAMPLE the optimal state
# ==============================================================================

def solve_vqa(qp, reps=3, maxiter=300, seed=42, shots=4096):
    rng = np.random.default_rng(seed)

    # ---- cost operator -------------------------------------------------------
    hamiltonian, offset, qubo = qubo_to_hamiltonian(qp)
    n = hamiltonian.num_qubits
    print(f"Hamiltonian on {n} qubits, {len(hamiltonian)} Pauli terms, offset={offset:.4f}")

    # ---- Steps 1 & 2: REFERENCE STATE + ANSATZ -------------------------------
    # QAOAAnsatz prepends the uniform superposition (reference state) and then
    # alternates the cost-layer exp(-i*gamma*H) with a mixer exp(-i*beta*X).
    # 'reps' = p = number of QAOA layers (depth vs. quality trade-off).
    ansatz = QAOAAnsatz(cost_operator=hamiltonian, reps=reps)
    ansatz.measure_all()  # add measurements so the Sampler can read bitstrings later

    estimator = StatevectorEstimator(seed=seed)
    sampler = StatevectorSampler(seed=seed)

    # The Estimator needs a circuit WITHOUT measurements; keep an unmeasured copy.
    ansatz_for_energy = QAOAAnsatz(cost_operator=hamiltonian, reps=reps)

    # ---- Step 4 (bootstrapping): INITIAL POINT -------------------------------
    # A standard warm start for QAOA: small gammas, betas near pi/4. You could
    # instead bootstrap from a cheaper low-reps run (the course's recommendation).
    num_params = ansatz_for_energy.num_parameters
    x0 = rng.uniform(0, np.pi, size=num_params) * 0.1

    # ---- Step 4: the OPTIMIZATION LOOP --------------------------------------
    # COBYLA: gradient-free, robust to the shot/statevector noise of the cost
    # oracle -- the course's default for this kind of landscape.
    history = {"n": 0}

    def logged_cost(p):
        e = cost_func(p, ansatz_for_energy, hamiltonian, estimator)
        history["n"] += 1
        if history["n"] % 25 == 0:
            print(f"  iter {history['n']:>4}: <H> = {e:+.5f}  (objective = {e + offset:+.5f})")
        return e

    print(f"\nOptimizing {num_params} QAOA angles with COBYLA (maxiter={maxiter})...")
    res = minimize(logged_cost, x0, method="COBYLA", options={"maxiter": maxiter})
    print(f"Converged: <H> = {res.fun:+.5f}, objective = {res.fun + offset:+.5f}")

    # ---- READOUT: sample the optimized state, take the most probable bitstring
    optimized = ansatz.assign_parameters(res.x)
    counts = sampler.run([optimized], shots=shots).result()[0].data.meas.get_counts()
    best_bitstring = max(counts, key=counts.get)

    # Qiskit bitstrings are little-endian (qubit 0 is the rightmost char).
    # qubo.variables is in the same order as our blocks list.
    bits = best_bitstring[::-1]
    x_star = {var.name: int(bits[i]) for i, var in enumerate(qubo.variables)}

    return x_star, res.fun + offset, counts


# ==============================================================================
# Reporting
# ==============================================================================

def report(blocks, x_star, objective):
    pruned = [b for b in blocks if x_star[b.name] == 1]
    kept = [b for b in blocks if x_star[b.name] == 0]
    total = sum(b.params for b in blocks)
    saved = sum(b.params for b in pruned)

    print(f"\n{'=' * 66}\nVQA / QAOA solution   (objective = {objective:+.4f})\n{'=' * 66}")
    print(f"PRUNE ({len(pruned)}):")
    for b in pruned:
        print(f"  - {b.name:<22s} params={b.params:>9,}  L_i={b.L:.4f}  C_i={b.C:.4f}")
    print(f"KEEP  ({len(kept)}):")
    for b in kept:
        flag = "  <-- protected" if b.L >= 0.5 else ""
        print(f"  - {b.name:<22s} params={b.params:>9,}  L_i={b.L:.4f}  C_i={b.C:.4f}{flag}")
    print(f"\nparams removed: {saved:,} / {total:,} ({100*saved/max(total,1):.1f}%)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="cost_loss_table.csv")
    ap.add_argument("--lambda-loss", type=float, default=8.0)
    ap.add_argument("--gamma-cost", type=float, default=1.0)
    ap.add_argument("--coupling", type=float, default=0.6)
    ap.add_argument("--reps", type=int, default=3, help="QAOA depth p")
    ap.add_argument("--maxiter", type=int, default=300)
    args = ap.parse_args()

    blocks = load_blocks(args.csv)
    J = build_coupling(blocks, coupling=args.coupling)
    qp = build_qubo(blocks, J, lambda_loss=args.lambda_loss, gamma_cost=args.gamma_cost)

    # Cross-check: the hand-built Hamiltonian must equal qiskit's auto-conversion.
    auto_H, auto_off, _ = qubo_to_hamiltonian(qp)
    hand_H, hand_off = build_hamiltonian_by_hand(
        blocks, J, lambda_loss=args.lambda_loss, gamma_cost=args.gamma_cost)
    match = (auto_H.simplify() == hand_H) and abs(auto_off - hand_off) < 1e-9
    print(f"Hand-derived Hamiltonian matches qiskit to_ising(): {match}")

    x_star, objective, counts = solve_vqa(qp, reps=args.reps, maxiter=args.maxiter)
    report(blocks, x_star, objective)


if __name__ == "__main__":
    main()
